In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# ==========================================
# CONFIGURATION
# ==========================================
NUM_ROWS = 20000
YEAR = 2025

# Centers & THEIR PERSONALITIES
# We define a specific 'base failure probability' for each center to ensure variety.
CENTERS_CONFIG = {
    'C001': {'name': 'Delhi',      'base_prob': 0.10}, # Standard
    'C002': {'name': 'Mumbai',     'base_prob': 0.14}, # Chaotic/Busy
    'C003': {'name': 'Kolkata',    'base_prob': 0.22}, # The Problem
    'C004': {'name': 'Bengaluru',  'base_prob': 0.07}, # Efficient
    'C005': {'name': 'Chandigarh', 'base_prob': 0.04}  # The Star
}

CENTERS = list(CENTERS_CONFIG.keys())

# Test Types
TEST_TYPES = {
    'T01': {'name': 'CBC (Blood)', 'tat': 60},
    'T02': {'name': 'X-Ray Chest', 'tat': 120},
    'T03': {'name': 'MRI Brain', 'tat': 240},
    'T04': {'name': 'Lipid Profile', 'tat': 90},
    'T05': {'name': 'COVID RT-PCR', 'tat': 360}
}

STATUS_OPTS = ['S05', 'S06', 'S03']
STATUS_WEIGHTS = [0.94, 0.04, 0.02] 

PATIENT_IDS = [f'P{str(i).zfill(5)}' for i in range(1, 3501)]

# ==========================================
# GENERATION LOGIC
# ==========================================
data = []

print(f"Generating {NUM_ROWS} rows with varied center performance...")

for i in range(1, NUM_ROWS + 1):
    test_id = f"TEST-{str(i).zfill(7)}"
    
    # 1. Date & Seasonality
    month_weights = [0.6, 0.6, 0.7, 0.7, 0.8, 1.0, 1.5, 1.5, 1.4, 0.9, 0.8, 0.7]
    month = random.choices(range(1, 13), weights=month_weights)[0]
    
    if month == 2: max_day = 28
    elif month in [4, 6, 9, 11]: max_day = 30
    else: max_day = 31
    day = random.randint(1, max_day)
    test_date = datetime(YEAR, month, day)
    
    is_peak_season = month in [7, 8, 9]

    # 2. Location Selection
    proc_center = np.random.choice(CENTERS, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    
    # 3. Mismatch Logic
    if random.random() < 0.15:
        actual_center = random.choice([c for c in CENTERS if c != proc_center])
        match_flag = "Mismatch"
    else:
        actual_center = proc_center
        match_flag = "Match"

    # 4. Test Type
    t_id = random.choice(list(TEST_TYPES.keys()))
    expected_tat = TEST_TYPES[t_id]['tat']
    
    # 5. Status
    status = np.random.choice(STATUS_OPTS, p=STATUS_WEIGHTS)
    
    # ---------------------------------------------------------
    # BREACH DETERMINATION (Using Center Personality)
    # ---------------------------------------------------------
    
    # Get the specific base probability for this center
    center_base = CENTERS_CONFIG[proc_center]['base_prob']
    
    # Start with that base
    breach_prob = center_base 
    
    # Add noise so it's not perfectly static per center
    breach_prob += random.uniform(-0.02, 0.02)
    
    # Modifier: Season
    if is_peak_season: breach_prob += 0.05
    
    # Modifier: Mismatch (Big penalty)
    if match_flag == "Mismatch": breach_prob += 0.25
    
    # Modifier: Complex Tests
    if t_id in ['T03', 'T05']: breach_prob += 0.03
    
    # Clamp
    breach_prob = max(0.01, min(1.0, breach_prob))
    
    # Roll the dice
    is_breach = random.random() < breach_prob
    
    # ---------------------------------------------------------
    # TIME GENERATION
    # ---------------------------------------------------------
    
    if is_breach:
        excess_factor = random.uniform(1.1, 2.2) 
        total_tat_min = int(expected_tat * excess_factor)
    else:
        safe_factor = random.uniform(0.5, 0.95)
        total_tat_min = int(expected_tat * safe_factor)
        if total_tat_min < 15: total_tat_min = 15
        
    # Distribute into Stages
    r1, r2, r3 = 0.25, 0.60, 0.15
    
    if match_flag == "Mismatch":
        r1, r2, r3 = 0.50, 0.40, 0.10
        
    if proc_center == 'C003' and not match_flag == "Mismatch":
        r1, r2, r3 = 0.20, 0.70, 0.10
        
    r1 += random.uniform(-0.05, 0.05)
    r2 += random.uniform(-0.05, 0.05)
    r3 += random.uniform(-0.05, 0.05)
    
    total_r = r1 + r2 + r3
    s1_dur = int(total_tat_min * (r1/total_r))
    s2_dur = int(total_tat_min * (r2/total_r))
    s3_dur = total_tat_min - s1_dur - s2_dur 
    
    s1_dur = max(5, s1_dur)
    s2_dur = max(5, s2_dur)
    s3_dur = max(5, s3_dur)
    
    # Timestamps
    start_min_offset = random.randint(480, 1080)
    stage1_start = datetime.combine(test_date, datetime.min.time()) + timedelta(minutes=start_min_offset)
    stage1_end = stage1_start + timedelta(minutes=s1_dur)
    stage2_start = stage1_end + timedelta(minutes=random.randint(2, 10)) 
    stage2_end = stage2_start + timedelta(minutes=s2_dur)
    stage3_start = stage2_end + timedelta(minutes=random.randint(2, 10))
    stage3_end = stage3_start + timedelta(minutes=s3_dur)
    
    if status == 'S06' or status == 'S03': 
        out_tat_flag = False
        under_tat_flag = False
        tat_missing = True
        total_tat_final = 0 
    else:
        tat_missing = False
        total_tat_final = total_tat_min
        if total_tat_final > expected_tat:
            out_tat_flag = True
            under_tat_flag = False
        else:
            out_tat_flag = False
            under_tat_flag = True

    row = [
        test_id,
        test_date.strftime('%Y-%m-%d'),
        random.choice(PATIENT_IDS),
        proc_center,
        actual_center,
        t_id,
        status,
        stage1_start.strftime('%Y-%m-%d %H:%M'),
        stage1_end.strftime('%Y-%m-%d %H:%M'),
        stage2_start.strftime('%Y-%m-%d %H:%M'),
        stage2_end.strftime('%Y-%m-%d %H:%M'),
        stage3_start.strftime('%Y-%m-%d %H:%M'),
        stage3_end.strftime('%Y-%m-%d %H:%M'),
        total_tat_final,
        expected_tat,
        under_tat_flag,
        out_tat_flag,
        tat_missing,
        match_flag
    ]
    data.append(row)

columns = [
    "Test_ID", "Test_Date", "Patient_ID", "Processing_Center_ID", 
    "Actual_Test_Center_ID", "TestType_ID", "Status_ID", 
    "Stage1_Start", "Stage1_End", "Stage2_Start", "Stage2_End", 
    "Stage3_Start", "Stage3_End", "Total_TAT_Minutes", 
    "Expected_TAT_Minutes", "Under_TAT_Flag", "Out_TAT_Flag", 
    "TAT_Missing_Flag", "Center_Match_Flag"
]

df = pd.DataFrame(data, columns=columns)

# Final Validity Check
breach_rate = df[df['Out_TAT_Flag'] == True].shape[0] / df.shape[0]
print(f"Dataset Generated Successfully.")
print(f"Final Overall Breach Rate: {breach_rate:.2%}")

# Save
filename = "Fact_Test_Performance_Final_v2.csv"
df.to_csv(filename, index=False)
print(f"File saved as: {filename}")

Generating 20000 rows with varied center performance...
Dataset Generated Successfully.
Final Overall Breach Rate: 17.46%
File saved as: Fact_Test_Performance_Final_v2.csv
